# 1. Feature Leakage

## What is Feature Leakage?

Feature leakage occurs when a machine learning model uses information during training that would not be available when making a real-world prediction.

This causes the model to learn from information it should not have access to.

As a result, the model may achieve very high accuracy during testing but perform poorly in real-world applications.

### Example: Customer Churn Prediction

Imagine that a telecom company wants to predict whether a customer will leave its service.

The company has the following information:

- Customer age
- Customer tenure
- Monthly charges
- Number of support calls
- Contract type
- Cancellation confirmation date

Customer age, tenure, monthly charges, support calls, and contract type may be available before predicting churn.

However, the cancellation confirmation date is recorded only after the customer decides to leave.

Using this cancellation information to predict churn would cause feature leakage.

## Types of Data Leakage

### 1. Target Leakage

Target leakage happens when an input feature contains information directly related to the target that would only become available after the target event.

Example:
- Target: Churn
- Feature: CancellationConfirmed

If cancellation is confirmed only after a customer leaves, this feature must not be used for prediction.

### 2. Train-Test Contamination

Train-test contamination occurs when information from the test dataset influences the training process.

Example:

- Scaling the complete dataset before splitting it.
- Selecting features using the complete dataset.
- Calculating missing-value replacements using both training and testing data.

The model indirectly learns information from the test dataset.

### 3. Temporal Leakage

Temporal leakage happens when information from the future is used to predict an earlier event.

Example:

A company wants to predict customer churn on November 1.

Using customer activity recorded on November 15 would introduce future information.

## Why is Feature Leakage Dangerous?

1. Produces misleadingly high accuracy.
2. Makes the model unreliable in production.
3. Causes incorrect business decisions.
4. Creates a difference between testing performance and real-world performance.
5. Reduces trust in machine learning predictions.

## How to Prevent Feature Leakage

1. Understand when each feature becomes available.
2. Split the dataset before fitting preprocessing transformations.
3. Fit scalers and imputers only on training data.
4. Use pipelines for preprocessing and model training.
5. Avoid using future information.
6. Remove features created after the target event.
7. Use time-based splitting when working with time-dependent data.

## Important Rule

A feature should be used only when its information is genuinely available at the time the prediction is made.

# 2. Demonstrate Feature Leakage

## Business Scenario: Telecom Customer Churn Prediction

A telecom company wants to identify customers who are likely to leave its services.

The company collects the following information:

| Feature | Description | Available Before Prediction? |
|---|---|---|
| Tenure | Number of months the customer has stayed | Yes |
| MonthlyCharges | Customer's monthly bill | Yes |
| SupportCalls | Number of support calls | Yes |
| ContractType | Customer's contract category | Yes |
| CancellationConfirmed | Cancellation confirmation after the customer decides to leave | No |
| Churn | Whether the customer left the company | Target |

## What Happens When Leakage Occurs?

Suppose the model uses `CancellationConfirmed` as an input feature.

This feature directly reveals whether the customer has churned.

The model may achieve almost 100% accuracy because it receives information that already tells it the answer.

However, this model cannot be used to predict future churn because cancellation confirmation is not available before the customer leaves.

## Correct Approach

We should train the model using only features available before the prediction:

- Tenure
- MonthlyCharges
- SupportCalls
- ContractType

We should remove `CancellationConfirmed` because it contains information from after the target event.

## Expected Learning

We will compare two models:

1. Model with safe features only.
2. Model with safe features and a leaked feature.

We will observe how leakage can create misleadingly high accuracy.

**Important:** This is a controlled demonstration. In real projects, leakage is not always this obvious.

In [1]:
# Step 1: Import Required Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Libraries imported successfully!")
# Step 2: Create a Synthetic Telecom Dataset

np.random.seed(42)

n = 500

data = pd.DataFrame({
    "CustomerID": range(1, n + 1),
    "Tenure": np.random.randint(1, 73, n),
    "MonthlyCharges": np.random.uniform(20, 120, n).round(2),
    "SupportCalls": np.random.randint(0, 10, n),
    "ContractType": np.random.choice(
        ["Monthly", "Yearly", "Two Year"],
        n,
        p=[0.5, 0.3, 0.2]
    )
})

# Create churn probabilities using customer information

churn_probability = (
    0.15
    + (data["Tenure"] < 12) * 0.20
    + (data["MonthlyCharges"] > 90) * 0.15
    + (data["SupportCalls"] > 5) * 0.20
    + (data["ContractType"] == "Monthly") * 0.15
)

churn_probability = churn_probability.clip(0, 0.95)

# Generate target variable

data["Churn"] = np.random.binomial(
    1,
    churn_probability
)

# Create a deliberately leaked feature.
# This information is recorded only after the customer churns.

data["CancellationConfirmed"] = data["Churn"]

print("Dataset created successfully!")

print("\nFirst Five Records:")
print(data.head())

print("\nDataset Shape:")
print(data.shape)

print("\nChurn Distribution:")
print(data["Churn"].value_counts())

# Step 3: Prepare the Dataset

# Convert categorical feature into numerical columns

data_encoded = pd.get_dummies(
    data,
    columns=["ContractType"],
    dtype=int
)

# Safe features available before prediction

safe_features = [
    "Tenure",
    "MonthlyCharges",
    "SupportCalls",
    "ContractType_Monthly",
    "ContractType_Yearly",
    "ContractType_Two Year"
]

X_safe = data_encoded[safe_features]

# Target variable

y = data_encoded["Churn"]

# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X_safe,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Dataset Shape:", X_train.shape)
print("Testing Dataset Shape:", X_test.shape)

# Step 4: Train Model Using Safe Features

safe_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

safe_model.fit(X_train, y_train)

safe_predictions = safe_model.predict(X_test)

safe_accuracy = accuracy_score(
    y_test,
    safe_predictions
)

print("Model Accuracy Without Leakage:")
print(round(safe_accuracy, 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    safe_predictions,
    zero_division=0
))

# Step 5: Train Model Using Safe Features + Leaked Feature

leaked_features = safe_features + [
    "CancellationConfirmed"
]

X_leaked = data_encoded[leaked_features]

X_train_leaked, X_test_leaked, y_train_leaked, y_test_leaked = (
    train_test_split(
        X_leaked,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

leaked_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

leaked_model.fit(
    X_train_leaked,
    y_train_leaked
)

leaked_predictions = leaked_model.predict(
    X_test_leaked
)

leaked_accuracy = accuracy_score(
    y_test_leaked,
    leaked_predictions
)

print("Model Accuracy With Leakage:")
print(round(leaked_accuracy, 4))

# Step 6: Compare Both Models

comparison = pd.DataFrame({
    "Model": [
        "Without Leakage",
        "With Leakage"
    ],
    "Accuracy": [
        safe_accuracy,
        leaked_accuracy
    ]
})

print(comparison)

print("\nAccuracy Difference:")
print(round(leaked_accuracy - safe_accuracy, 4))

# Step 7: Demonstrate Correct Preprocessing Using a Pipeline

# A pipeline fits the scaler only on the training data.
# The same fitted scaler is then applied to the test data.

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

pipeline_predictions = pipeline.predict(X_test)

pipeline_accuracy = accuracy_score(
    y_test,
    pipeline_predictions
)

print("Pipeline Model Accuracy:")
print(round(pipeline_accuracy, 4))

# Step 8: Final Feature Leakage Analysis

feature_analysis = pd.DataFrame({
    "Feature": [
        "Tenure",
        "MonthlyCharges",
        "SupportCalls",
        "ContractType",
        "CancellationConfirmed"
    ],
    "Available_Before_Prediction": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "No"
    ],
    "Decision": [
        "Retain",
        "Retain",
        "Retain",
        "Retain",
        "Remove"
    ]
})

print("Feature Leakage Analysis:")
print(feature_analysis)

Libraries imported successfully!
Dataset created successfully!

First Five Records:
   CustomerID  Tenure  MonthlyCharges  SupportCalls ContractType  Churn  \
0           1      52           57.63             5       Yearly      0   
1           2      15           28.35             7     Two Year      1   
2           3      72           97.71             1      Monthly      1   
3           4      61           75.84             8       Yearly      0   
4           5      21           62.42             4       Yearly      0   

   CancellationConfirmed  
0                      0  
1                      1  
2                      1  
3                      0  
4                      0  

Dataset Shape:
(500, 7)

Churn Distribution:
Churn
0    315
1    185
Name: count, dtype: int64
Training Dataset Shape: (400, 6)
Testing Dataset Shape: (100, 6)
Model Accuracy Without Leakage:
0.63

Classification Report:
              precision    recall  f1-score   support

           0       0.70   

# 4. Feature Justification

## Feature Analysis Table

| Feature Name | Source Columns | Feature Logic | Reason for Creation | Expected Relationship with Churn | ML Value | Leakage Risk | Final Decision |
|---|---|---|---|---|---|---|---|
| Tenure | Tenure | Original customer tenure | Understand customer relationship duration | Lower tenure may increase churn | High | Low | Retain |
| MonthlyCharges | MonthlyCharges | Original monthly charges | Understand customer spending | Higher charges may increase churn | Medium | Low | Retain |
| SupportCalls | SupportCalls | Original support call count | Understand customer service problems | More calls may increase churn | High | Low | Retain |
| ContractType | ContractType | One-hot encoding | Represent contract categories numerically | Monthly contracts may have higher churn | High | Low | Retain |
| CancellationConfirmed | Churn | Directly copies the churn outcome | Demonstrate target leakage | Directly reveals churn | Invalid for prediction | Very High | Remove |

## Why Should CancellationConfirmed Be Removed?

`CancellationConfirmed` contains information recorded after the customer has already decided to leave.

If we use this feature, the model receives information about the target before making its prediction.

This creates an unrealistic model with misleading accuracy.

Therefore, it must be removed from the final machine learning dataset.

## Before and After Leakage Prevention

### Before Leakage Prevention

- Model uses both valid and leaked features.
- Accuracy becomes artificially high.
- Model depends on information unavailable during prediction.

### After Leakage Prevention

- Model uses only information available before prediction.
- Evaluation becomes more realistic.
- Model is more suitable for real-world deployment.

## Final Learning

Feature leakage is one of the important problems in machine learning.

A model should learn from past and currently available information, not from future events or information that reveals the target.

**Final Rule:** Always check whether a feature is available at the exact time when the model needs to make its prediction.